<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    05 · Colección de Datos para Drug Discovery
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:620px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 2 — Sin experiencia previa en programación</em>
  </p>
</div>


---
## ¿Qué aprenderás en este notebook?

| # | Sección | Base de datos | Novedad frente a NB-00 |
|---|---------|--------------|------------------------|
| 1 | Instalación y contexto | — | — |
| 2 | ChEMBL: búsqueda avanzada de targets | ChEMBL | Filtros por organismo, tipo de target, UniProt |
| 3 | ChEMBL: múltiples tipos de actividad | ChEMBL | IC50, Ki, EC50, Kd — descarga simultánea |
| 4 | ChEMBL: información de compuestos | ChEMBL | Propiedades ADME, fases clínicas, mecanismo de acción |
| 5 | ChEMBL: assays y documentos | ChEMBL | Tipos de ensayo, fuentes bibliográficas |
| 6 | ChEMBL: búsqueda por similitud de estructura | ChEMBL | Similitud SMILES, substructure search |
| 7 | PubChem: búsqueda de compuestos | PubChem | API PUG-REST, propiedades, sinónimos |
| 8 | PDB: estructuras 3D de proteínas | PDB | Descargar PDB, metadatos, ligandos co-cristalizados |
| 9 | Cruce de bases de datos | ChEMBL+PubChem+PDB | Conectar información entre las tres fuentes |
| 10 | Ejercicio integrador | — | Colectar un dataset completo para tu target |

---
> **Nota:** En el NB-00 viste una primera consulta a ChEMBL (búsqueda de BCR-ABL, IC50 básico).  
> En este notebook profundizamos mucho más en ChEMBL y añadimos PubChem y PDB.


---
## 1. Instalación de librerías

En este notebook usaremos tres librerías principales:

- `chembl_webresource_client` — cliente oficial de ChEMBL para Python
- `pubchempy` — cliente de PubChem para Python  
- `requests` + `rcsbsearchapi` — para consultar el PDB (Protein Data Bank)
- `pandas`, `matplotlib` — manejo de datos y visualización


In [ ]:
# ── Instalar librerías (solo la primera vez en Colab) ───────────────────────
!pip install chembl_webresource_client pubchempy rcsbsearchapi --quiet

print("✅ Librerías instaladas")


In [ ]:
# ── Importaciones globales ──────────────────────────────────────────────────
from chembl_webresource_client.new_client import new_client
import pubchempy as pcp
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

print("✅ Todo listo")
print()
print("Bases de datos que usaremos hoy:")
print("  🔵 ChEMBL  — bioactividades, targets, moléculas drug-like")
print("  🟠 PubChem — compuestos químicos, propiedades, sinónimos")
print("  🟢 PDB     — estructuras 3D de proteínas y complejos")


---
## 2. ChEMBL: búsqueda avanzada de targets

En el NB-00 buscamos un target por nombre. Aquí vamos más profundo:
- Filtrar por **organismo** (solo targets humanos)
- Filtrar por **tipo de target** (proteína, receptor, enzima...)
- Obtener el **ID de UniProt** de un target para cruzarlo con otras bases de datos
- Explorar **qué información contiene** cada registro de target


In [ ]:
# ── Conectar con el cliente de targets de ChEMBL ────────────────────────────
target_client = new_client.target

# ── Búsqueda básica por nombre ───────────────────────────────────────────────
resultados = target_client.search('EGFR')
df_targets = pd.DataFrame.from_records(resultados)

print(f"Resultados para 'EGFR': {len(df_targets)} targets encontrados")
print()
print("Columnas disponibles en un registro de target:")
print(df_targets.columns.tolist())


In [ ]:
# ── Ver la información completa de las primeras entradas ────────────────────
columnas_target = [
    'target_chembl_id',
    'pref_name',
    'target_type',
    'organism',
    'species_group_flag',
]
cols = [c for c in columnas_target if c in df_targets.columns]
print(df_targets[cols].head(12).to_string(index=False))


In [ ]:
# ── Filtrar: solo targets humanos de tipo proteína ──────────────────────────
# ChEMBL permite filtrar directamente en la consulta, sin descargar todo

egfr_humano = target_client.filter(
    pref_name__icontains='epidermal growth factor',   # nombre contiene (case-insensitive)
    organism='Homo sapiens',
    target_type='SINGLE PROTEIN'
)
df_egfr = pd.DataFrame.from_records(egfr_humano)

print(f"EGFR humano (proteína única): {len(df_egfr)} resultado(s)")
print()
if len(df_egfr) > 0:
    cols = [c for c in columnas_target if c in df_egfr.columns]
    print(df_egfr[cols].to_string(index=False))


In [ ]:
# ── Ver los tipos de target disponibles en ChEMBL ──────────────────────────
todos_tipos = target_client.filter(organism='Homo sapiens').only([
    'target_chembl_id', 
    'pref_name', 
    'target_type'
])
df_tipos = pd.DataFrame.from_records(todos_tipos)

if 'target_type' in df_tipos.columns:
    conteo = df_tipos['target_type'].value_counts().head(12)
    print("TIPOS DE TARGET HUMANO EN ChEMBL (Top 12)")
    print("=" * 45)
    for tipo, n in conteo.items():
        barra = '█' * min(int(n / conteo.max() * 25), 25)
        print(f"  {tipo:<30} {n:>5}  {barra}")


In [ ]:
# ── Obtener el ID de UniProt de un target ──────────────────────────────────
# UniProt es la base de datos canónica de secuencias proteicas.
# El ID de UniProt permite cruzar ChEMBL con PDB, AlphaFold, etc.

TARGET_ID = 'CHEMBL203'   # EGFR (Homo sapiens)
info = target_client.get(TARGET_ID)

print("=" * 55)
print(f"Target: {info['pref_name']}")
print(f"ChEMBL ID: {info['target_chembl_id']}")
print(f"Organismo: {info['organism']}")
print(f"Tipo: {info['target_type']}")
print()

# Los componentes del target contienen el acceso a UniProt
if 'target_components' in info and info['target_components']:
    print("COMPONENTES (secuencias proteicas):")
    for comp in info['target_components']:
        accession = comp.get('accession', 'N/A')
        print(f"  UniProt accession: {accession}")
        print(f"  → https://www.uniprot.org/uniprot/{accession}")
        print(f"  → Estructura PDB:  https://www.rcsb.org/search?q={accession}")


---
## 3. ChEMBL: descarga de múltiples tipos de actividad

En el 02 descargamos solo IC50. En realidad ChEMBL reporta muchos tipos de actividad.  
Entender cuándo usar cada uno es fundamental para la **curación de datos** (Semana 3).


In [ ]:
# ── Tipos de actividad más comunes en ChEMBL ────────────────────────────────
activity_client = new_client.activity

print("TIPOS DE ACTIVIDAD BIOLÓGICA EN ChEMBL")
print("=" * 60)
print()
tipos_actividad = {
    'IC50':  ('Inhibitory Concentration 50%',
              'Concentración que inhibe el 50% de la actividad del target',
              'Inhibidores — más común en drug discovery'),
    'Ki':    ('Inhibition Constant',
              'Constante de equilibrio de inhibición (termodinámica)',
              'Medida más rigurosa de afinidad — independiente del sustrato'),
    'EC50':  ('Effective Concentration 50%',
              'Concentración que produce el 50% del efecto máximo',
              'Agonistas y activadores — no inhibidores'),
    'Kd':    ('Dissociation Constant',
              'Constante de disociación del complejo proteína-ligando',
              'Medida biofísica — SPR, ITC, fluorescencia'),
    'GI50':  ('Growth Inhibition 50%',
              'Concentración que inhibe el 50% del crecimiento celular',
              'Ensayos de viabilidad celular — oncología'),
    'MIC':   ('Minimum Inhibitory Concentration',
              'Concentración mínima que inhibe el crecimiento microbiano',
              'Antibióticos y antifúngicos'),
}

for tipo, (nombre, descripcion, uso) in tipos_actividad.items():
    print(f"  {tipo:<8} {nombre}")
    print(f"           {descripcion}")
    print(f"           ↳ Uso: {uso}")
    print()


In [ ]:
# ── Optimización: Descarga Única y Filtrado Local ──────────────────────────
TARGET_ID = 'CHEMBL203'

print(f"Descargando todos los Binding Assays para {TARGET_ID}...")

# La descarga desde ChEMBL de esta información demora mas de 40 min, por lo cual cargaremos un archivo pre descargado

# # Descarga de los datos
# query = activity_client.filter(
#     target_chembl_id=TARGET_ID,
#     assay_type='B'
# ).only([
#     'molecule_chembl_id', 'canonical_smiles', 
#     'standard_type', 'standard_value', 'standard_units', 'pchembl_value'
# ])

# df_todo = pd.DataFrame.from_records(query)

df_todo = pd.read_csv('D:\\data_git\\curso_datascience\\files\\chembl_203.csv')

cols_num = ['standard_value', 'pchembl_value']
for col in cols_num:
    df_todo[col] = pd.to_numeric(df_todo[col], errors='coerce')

tipos_interes = ['IC50', 'Ki', 'Kd']
df_filtrado = df_todo[df_todo['standard_type'].isin(tipos_interes)].copy()

print("\nResumen de datos locales:")
for tipo in tipos_interes:
    conteo = len(df_filtrado[df_filtrado['standard_type'] == tipo])
    print(f"  {tipo:<6}: {conteo:>5} registros")

print(f"\nTotal filtrado: {len(df_filtrado)} de {len(df_todo)} registros totales.")


In [ ]:
# ── Comparar distribuciones de pIC50, pKi, pKd ─────────────────────────────
# pValor = -log10(Valor en Molar) → más negativo el valor, mayor la potencia

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colores = {'IC50': '#e94560', 'Ki': '#4a90d9', 'Kd': '#27ae60'}

for ax, tipo in zip(axes, ['IC50', 'Ki', 'Kd']):    
    df = df_filtrado[df_filtrado['standard_type'] == tipo]
    if 'pchembl_value' in df.columns:
        datos_validos = df['pchembl_value'].dropna()
        if len(datos_validos) > 0:
            ax.hist(datos_validos, bins=30, color=colores[tipo],
                    alpha=0.8, edgecolor='white', linewidth=0.5)
            ax.axvline(datos_validos.median(), color='black',
                       linestyle='--', linewidth=1.5, label=f'Mediana: {datos_validos.median():.1f}')
            ax.set_title(f'p{tipo} — EGFR (n={len(datos_validos)})', fontsize=11)
            ax.set_xlabel(f'p{tipo} (-log₁₀M)', fontsize=10)
            ax.set_ylabel('Frecuencia', fontsize=10)
            ax.legend(fontsize=9)
            ax.set_xlim(3, 12)

plt.suptitle('Distribución de actividades para EGFR (ChEMBL)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()

ruta_objetivo = '../figures'

if not os.path.exists(ruta_objetivo):
    os.makedirs(ruta_objetivo)
    print(f"✅ Carpeta creada exitosamente en: {ruta_objetivo}")
else:
    print("ℹ️ La carpeta ya existe, no fue necesario crearla.")

plt.savefig(f'{ruta_objetivo}/distribucion_actividades_EGFR.png', dpi=120, bbox_inches='tight')
plt.show()
print("💡 Un pIC50 de 9 = IC50 de 1 nM (muy potente)")
print("   Un pIC50 de 6 = IC50 de 1 µM (moderado)")
print("   Un pIC50 de 5 = IC50 de 10 µM (débil)")


In [ ]:
# ── También existen ensayos funcionales (assay_type='F') ───────────────────
# DIFERENCIA IMPORTANTE:
#   assay_type='B' → Binding: mide unión directa (física) proteína-ligando
#   assay_type='F' → Functional: mide efecto funcional (ej. fosforilación)
#   assay_type='A' → ADME: permeabilidad, metabolismo, solubilidad
#   assay_type='T' → Toxicidad: citotoxicidad, genotoxicidad

datos_func = activity_client.filter(
    target_chembl_id=TARGET_ID,
    standard_type='IC50',
    assay_type='F'
)
df_func = pd.DataFrame.from_records(datos_func)

print("TIPOS DE ENSAYO DISPONIBLES PARA EGFR")
print("=" * 45)
print(f"  Binding (B):    {len(df_filtrado[df_filtrado['standard_type'] == 'IC50']):>5} registros IC50")
print(f"  Functional (F): {len(df_func):>5} registros IC50")
print()
print("⚠️  Semana 3: No mezclaremos Binding y Functional")
print("   sin un análisis previo — pueden dar valores muy diferentes")
print("   para el mismo compuesto.")


---
## 4. ChEMBL: información detallada de compuestos

ChEMBL no es solo una base de datos de bioactividad — también contiene información  
sobre las propiedades fisicoquímicas, fases clínicas y mecanismos de acción de las moléculas.


In [ ]:
# ── Buscar un compuesto por nombre ──────────────────────────────────────────
molecule_client = new_client.molecule

# Buscar erlotinib (inhibidor de EGFR aprobado)
resultado = molecule_client.search('erlotinib')
df_mol = pd.DataFrame.from_records(resultado)

print(f"Resultados para 'erlotinib': {len(df_mol)}")
print()
print("Columnas disponibles en un registro de molécula:")
for col in df_mol.columns.tolist():
    print(f"  {col}")


In [ ]:
# ── Propiedades fisicoquímicas (drug-likeness) ──────────────────────────────
# Buscar el erlotinib exacto
erlotinib = molecule_client.filter(pref_name='ERLOTINIB')[0]

print("=" * 55)
print(f"Molécula: {erlotinib['pref_name']}")
print(f"ChEMBL ID: {erlotinib['molecule_chembl_id']}")
print(f"SMILES: {erlotinib.get('molecule_structures', {}).get('canonical_smiles', 'N/A')[:60]}")
print()

props = erlotinib.get('molecule_properties', {})
if props:
    print("PROPIEDADES FISICOQUÍMICAS (Regla de Lipinski):")
    print("-" * 45)
    mapa = {
        'mw_freebase':       ('Peso molecular (Da)',         500),
        'alogp':             ('LogP (lipofilia)',            5.0),
        'hbd':               ('Donadores de H-bond',        5),
        'hba':               ('Aceptores de H-bond',        10),
        'psa':               ('Área polar sup. (Å²)',        140),
        'rtb':               ('Rotatable bonds',            10),
        'ro5_violations':    ('Violaciones Regla 5',        0),
        'qed_weighted':      ('QED (drug-likeness 0-1)',    None),
        'aromatic_rings':    ('Anillos aromáticos',         None),
    }
    for key, (label, limite) in mapa.items():
        val = props.get(key)
        if val is not None:
            flag = ''
            if limite is not None:
                try:
                    flag = ' ✅' if float(val) <= limite else ' ⚠️'
                except: pass
            print(f"  {label:<30} {val}{flag}")


In [ ]:
# ── Fase clínica y estado de aprobación ─────────────────────────────────────
print("INFORMACIÓN CLÍNICA Y REGULATORIA")
print("=" * 45)

campos_clinicos = {
    'max_phase':             'Fase clínica máxima alcanzada',
    'first_approval':        'Año de primera aprobación',
    'oral':                  '¿Administración oral?',
    'parenteral':            '¿Administración parenteral?',
    'topical':               '¿Administración tópica?',
    'black_box_warning':     '¿Tiene peligros de caja negra (FDA)?', # Fármacos con riesgos muy altos
    'natural_product':       '¿Es producto natural?',
    'availability_type':     'Tipo de disponibilidad',
}

fases = {-1: 'Clínica fallida', 0: 'Preclínico',
          1: 'Fase I', 2: 'Fase II', 3: 'Fase III', 4: 'Aprobado'}

for campo, etiqueta in campos_clinicos.items():
    val = erlotinib.get(campo, 'N/A')
    if campo == 'max_phase' and val is not None:
        try: val = f"{val} ({fases.get(int(val), 'Desconocido')})"
        except: pass
    print(f"  {etiqueta:<35} {val}")


In [ ]:
# ── Indicaciones aprobadas (drug indications) ───────────────────────────────
drug_ind = new_client.drug_indication

indicaciones = drug_ind.filter(
    molecule_chembl_id=erlotinib['molecule_chembl_id']
)
df_ind = pd.DataFrame.from_records(indicaciones)

print("INDICACIONES APROBADAS DE ERLOTINIB")
print("=" * 55)
if len(df_ind) > 0:
    cols_ind = ['mesh_heading', 'max_phase_for_ind', 'efo_term']
    cols_ind = [c for c in cols_ind if c in df_ind.columns]
    print(df_ind[cols_ind].to_string(index=False))
else:
    print("No se encontraron indicaciones registradas.")
print()
print("💡 'max_phase_for_ind' = fase clínica para esa indicación específica")


In [ ]:
# ── Comparar múltiples inhibidores de EGFR aprobados ────────────────────────
inhibidores_egfr = ['ERLOTINIB', 'GEFITINIB', 'AFATINIB',
                    'OSIMERTINIB', 'LAPATINIB', 'DACOMITINIB']

datos_comparacion = []
for nombre in inhibidores_egfr:
    res = molecule_client.filter(pref_name=nombre)
    if res:
        mol = res[0]
        props = mol.get('molecule_properties', {}) or {}
        datos_comparacion.append({
            'Nombre':       mol.get('pref_name', nombre),
            'ChEMBL ID':    mol.get('molecule_chembl_id', ''),
            'Fase':         mol.get('max_phase', ''),
            'PM (Da)':      props.get('mw_freebase', ''),
            'LogP':         props.get('alogp', ''),
            'HBD':          props.get('hbd', ''),
            'HBA':          props.get('hba', ''),
            'QED':          props.get('qed_weighted', ''),
            'Aprobación':   mol.get('first_approval', ''),
        })

df_comp = pd.DataFrame(datos_comparacion)
print("COMPARACIÓN DE INHIBIDORES DE EGFR APROBADOS")
print("=" * 75)
print(df_comp.to_string(index=False))
print()
print("QED = Quantitative Estimate of Drug-likeness (0=peor, 1=mejor)")


---
## 5. ChEMBL: assays y documentos fuente

Cada dato de actividad en ChEMBL proviene de un **assay** publicado en un artículo científico.  
Entender el assay es crucial para evaluar la calidad y comparabilidad de los datos.


In [ ]:
# ── Explorar assays disponibles para EGFR ──────────────────────────────────
assay_client = new_client.assay

assays_egfr = assay_client.filter(
    target_chembl_id='CHEMBL203',
    assay_type='B'
)
df_assays = pd.DataFrame.from_records(assays_egfr)

print(f"Assays de tipo Binding para EGFR: {len(df_assays)}")
print()
print("Columnas disponibles:")
print(df_assays.columns.tolist())


In [ ]:
# ── Información de un assay específico ─────────────────────────────────────
if len(df_assays) > 0:
    cols_assay = ['assay_chembl_id', 'assay_type', 'assay_organism',
                  'confidence_score', 'description', 'document_chembl_id']
    cols_assay = [c for c in cols_assay if c in df_assays.columns]

    print("MUESTRA DE ASSAYS PARA EGFR (primeros 5)")
    print("=" * 70)
    for _, row in df_assays[cols_assay].head(5).iterrows():
        print(f"  ID:          {row.get('assay_chembl_id','')}")
        print(f"  Tipo:        {row.get('assay_type','')}")
        print(f"  Organismo:   {row.get('assay_organism','')}")
        print(f"  Confianza:   {row.get('confidence_score','')} (0-9, 9=mayor)")
        desc = str(row.get('description',''))
        print(f"  Descripción: {desc[:90]}{'...' if len(desc)>90 else ''}")
        print()

print("💡 Confidence score en ChEMBL:")
print("   9 = Target único y directo (gold standard)")
print("   7 = Target relacionado directamente")
print("   4 = Target inferido del assay")
print("   1 = Sin información suficiente")


In [ ]:
# ── Distribución de confidence scores ──────────────────────────────────────
if 'confidence_score' in df_assays.columns:
    scores = df_assays['confidence_score'].value_counts().sort_index()
    print("DISTRIBUCIÓN DE CONFIDENCE SCORES EN ASSAYS DE EGFR")
    print("=" * 50)
    for score, count in scores.items():
        barra = '█' * min(int(count / scores.max() * 30), 30)
        label = {9:'✅ Directo', 7:'🟡 Relacionado',
                 4:'🟠 Inferido', 1:'🔴 Sin info'}.get(int(score), '')
        print(f"  Score {score}: {count:>4}  {barra}  {label}")


In [ ]:
# ── Obtener el documento fuente de un assay ─────────────────────────────────
document_client = new_client.document

if len(df_assays) > 0 and 'document_chembl_id' in df_assays.columns:
    doc_id = df_assays['document_chembl_id'].dropna().iloc[0]
    doc = document_client.get(doc_id)

    print("DOCUMENTO FUENTE DE UN ASSAY")
    print("=" * 55)
    campos_doc = ['document_chembl_id', 'title', 'authors',
                  'journal', 'year', 'doi', 'pubmed_id']
    for campo in campos_doc:
        val = doc.get(campo, 'N/A')
        if isinstance(val, str) and len(val) > 80:
            val = val[:80] + '...'
        print(f"  {campo:<22} {val}")
    print()
    doi = doc.get('doi', '')
    if doi:
        print(f"  → Acceso al artículo: https://doi.org/{doi}")


---
## 6. ChEMBL: búsqueda por similitud de estructura y subestructura

Además de buscar por nombre o ID, ChEMBL permite buscar moléculas por su **estructura química**.  
Esto es fundamental en drug discovery para encontrar análogos de un compuesto de interés.


In [ ]:
# ── Búsqueda por similitud (similarity search) ──────────────────────────────
# Dado un SMILES de referencia, ChEMBL devuelve moléculas similares
# según la Similitud de Tanimoto sobre Morgan fingerprints

# SMILES del erlotinib
SMILES_ERLOTINIB = "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1"

similarity_client = new_client.similarity

print("Buscando moléculas similares al erlotinib en ChEMBL...")
print(f"  SMILES: {SMILES_ERLOTINIB}")
print(f"  Umbral de similitud: 70%")
print()

similares = similarity_client.filter(
    smiles=SMILES_ERLOTINIB,
    similarity=70          # porcentaje de similitud mínima (Tanimoto)
)
df_sim = pd.DataFrame.from_records(similares)

print(f"Moléculas similares encontradas: {len(df_sim)}")
print()
if len(df_sim) > 0:
    cols_sim = ['molecule_chembl_id', 'pref_name', 'similarity',
                'max_phase', 'molecule_structures']
    cols_sim = [c for c in cols_sim if c in df_sim.columns]
    df_vista = df_sim[cols_sim].copy()
    if 'similarity' in df_vista.columns:
        df_vista['similarity'] = pd.to_numeric(df_vista['similarity'], errors='coerce')
        df_vista = df_vista.sort_values('similarity', ascending=False)
    print(df_vista.head(10).to_string(index=False))


In [ ]:
# ── Búsqueda por subestructura (substructure search) ───────────────────────
# Encuentra todas las moléculas que contienen un fragmento estructural dado.
# Útil para buscar todos los compuestos con un scaffold (núcleo) específico.

substructure_client = new_client.substructure

# Buscamos el scaffold quinazolina — presente en erlotinib, gefitinib, afatinib
SCAFFOLD_QUINAZOLINA = "c1cnc2ccccc2n1"   # quinazolina básica

print("Buscando moléculas con scaffold quinazolina en ChEMBL...")
print(f"  SMARTS/SMILES: {SCAFFOLD_QUINAZOLINA}")
print("Tiempo mas de 6 horas de busqueda")
print()

subestructuras = substructure_client.filter(smiles=SCAFFOLD_QUINAZOLINA)
df_sub = pd.DataFrame.from_records(subestructuras)

print(f"Moléculas con scaffold quinazolina: {len(df_sub)}")
print()
if len(df_sub) > 0 and 'max_phase' in df_sub.columns:
    fases = df_sub['max_phase'].value_counts().sort_index(ascending=False)
    print("Distribución por fase clínica:")
    labels_fase = {4:'Aprobado', 3:'Fase III', 2:'Fase II',
                   1:'Fase I', 0:'Preclínico', -1:'Fallido'}
    for fase, count in fases.items():
        try: label = labels_fase.get(int(fase), str(fase))
        except: label = str(fase)
        print(f"  {label:<12}: {count:>4}")


---
## 7. PubChem: búsqueda de compuestos

**PubChem** es la base de datos química más grande del mundo, mantenida por el NCBI (NIH).  
Contiene más de 100 millones de sustancias. Es complementaria a ChEMBL:

| | ChEMBL | PubChem |
|--|--------|---------|
| **Fortaleza** | Bioactividad drug-like | Cobertura química total |
| **Datos clave** | IC50, Ki, mecanismo | Propiedades, sinónimos, espectros |
| **Tamaño** | ~2.4M compuestos | ~100M sustancias |
| **Uso principal** | Drug discovery | Referencia química general |


In [ ]:
# ── Buscar un compuesto en PubChem por nombre ───────────────────────────────
import pubchempy as pcp

# Buscar erlotinib
compuestos = pcp.get_compounds('erlotinib', 'name')

print(f"Resultados de PubChem para 'erlotinib': {len(compuestos)}")
print()

if compuestos:
    c = compuestos[0]
    print(f"CID (PubChem ID):    {c.cid}")
    print(f"Nombre IUPAC:        {c.iupac_name[:70] if c.iupac_name else 'N/A'}")
    print(f"Fórmula molecular:   {c.molecular_formula}")
    print(f"Peso molecular:      {c.molecular_weight}")
    print(f"SMILES canónico:     {c.canonical_smiles[:60] if c.canonical_smiles else 'N/A'}...")
    print(f"SMILES isométrico:   {c.isomeric_smiles[:60] if c.isomeric_smiles else 'N/A'}...")
    print(f"InChIKey:            {c.inchikey}")


In [ ]:
# ── Propiedades fisicoquímicas desde PubChem ────────────────────────────────
if compuestos:
    c = compuestos[0]
    print("PROPIEDADES FISICOQUÍMICAS (PubChem)")
    print("=" * 50)

    propiedades = {
        'XLogP':                    c.xlogp,
        'TPSA (Å²)':               c.tpsa,
        'Rotatable bonds':          c.rotatable_bond_count,
        'H-bond donors':            c.h_bond_donor_count,
        'H-bond acceptors':         c.h_bond_acceptor_count,
        'Heavy atom count':         c.heavy_atom_count,
        'Charge':                   c.charge,
        'Complexity':               c.complexity,
        'Exact mass':               c.exact_mass,
        'Isotope atom count':       c.isotope_atom_count,
        'Defined stereo centers':   c.defined_atom_stereo_count,
    }

    for prop, val in propiedades.items():
        print(f"  {prop:<30} {val}")


In [ ]:
# ── Sinónimos de un compuesto ───────────────────────────────────────────────
# PubChem recopila todos los nombres con los que se conoce un compuesto:
# nombre genérico, nombre comercial, nombre IUPAC, código de investigación...

if compuestos:
    c = compuestos[0]
    sinonimos = pcp.get_synonyms(c.cid, 'cid')

    print(f"SINÓNIMOS DE ERLOTINIB EN PubChem ({len(sinonimos[0]['Synonym'])} nombres)")
    print("=" * 55)
    for i, sin in enumerate(sinonimos[0]['Synonym'][:20]):
        print(f"  {i+1:>2}. {sin}")
    if len(sinonimos[0]['Synonym']) > 20:
        print(f"  ... y {len(sinonimos[0]['Synonym'])-20} más")


In [ ]:
# ── Buscar por SMILES en PubChem (identity search) ─────────────────────────
# PubChem también permite buscar por estructura

SMILES_ASPIRINA = "CC(=O)Oc1ccccc1C(=O)O"

compuestos_asp = pcp.get_compounds(SMILES_ASPIRINA, 'smiles')
if compuestos_asp:
    c_asp = compuestos_asp[0]
    print(f"Búsqueda por SMILES de aspirina en PubChem:")
    print(f"  CID:               {c_asp.cid}")
    print(f"  Fórmula:           {c_asp.molecular_formula}")
    print(f"  Peso molecular:    {c_asp.molecular_weight}")
    print(f"  InChIKey:          {c_asp.inchikey}")
    print()
    print(f"  → URL PubChem: https://pubchem.ncbi.nlm.nih.gov/compound/{c_asp.cid}")


In [ ]:
# ── Obtener múltiples propiedades de varios compuestos vía API REST ─────────
# La API PUG-REST de PubChem permite pedir propiedades específicas en batch

cids_egfr_inhibidores = {
    'Erlotinib':   176870,
    'Gefitinib':   123631,
    'Afatinib':    10184653,
    'Osimertinib': 71496458,
    'Lapatinib':   208908,
}

propiedades_request = 'MolecularFormula,MolecularWeight,XLogP,TPSA,HBondDonorCount,HBondAcceptorCount,RotatableBondCount'

resultados_pubchem = []
for nombre, cid in cids_egfr_inhibidores.items():
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/{propiedades_request}/JSON"
    r = requests.get(url, timeout=15)
    if r.status_code == 200:
        props = r.json()['PropertyTable']['Properties'][0]
        props['Nombre'] = nombre
        resultados_pubchem.append(props)

df_pubchem = pd.DataFrame(resultados_pubchem)
cols_orden = ['Nombre', 'MolecularFormula', 'MolecularWeight',
              'XLogP', 'TPSA', 'HBondDonorCount',
              'HBondAcceptorCount', 'RotatableBondCount']
cols_orden = [c for c in cols_orden if c in df_pubchem.columns]
df_pubchem = df_pubchem[cols_orden].set_index('Nombre')

print("PROPIEDADES DE INHIBIDORES DE EGFR — PubChem API")
print("=" * 70)
print(df_pubchem.to_string())


---
## 8. PDB: estructuras 3D de proteínas

El **Protein Data Bank (RCSB PDB)** es el repositorio global de estructuras 3D de macromoléculas biológicas.  
Contiene más de 220.000 estructuras obtenidas por cristalografía de rayos X, cryo-EM y RMN.

En drug discovery, las estructuras del PDB son esenciales para:
- **Docking molecular** (Semana 6 del curso)
- Entender el sitio de unión del target
- Visualizar cómo interacciona un fármaco con la proteína


In [ ]:
# ── Buscar estructuras de EGFR en el PDB via API REST ──────────────────────
# El PDB tiene una API REST en https://data.rcsb.org

def buscar_pdb_por_uniprot(uniprot_id, max_resultados=20):
    """
    Busca estructuras en el PDB para un UniProt accession dado.
    Devuelve un DataFrame con los resultados.
    """
    url = "https://search.rcsb.org/rcsbsearch/v2/query"

    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {"start": 0, "rows": max_resultados},
            "sort": [{"sort_by": "score", "direction": "desc"}]
        }
    }

    r = requests.post(url, json=query, timeout=15)
    if r.status_code == 200:
        data = r.json()
        ids = [hit['identifier'] for hit in data.get('result_set', [])]
        total = data.get('total_count', 0)
        return ids, total
    return [], 0

# UniProt de EGFR humano = P00533
UNIPROT_EGFR = 'P00533'
pdb_ids, total = buscar_pdb_por_uniprot(UNIPROT_EGFR, max_resultados=10)

print(f"Estructuras de EGFR (UniProt {UNIPROT_EGFR}) en el PDB:")
print(f"  Total disponibles: {total}")
print(f"  Primeros 10: {pdb_ids}")


In [ ]:
# ── Obtener metadatos de una estructura específica ──────────────────────────
def obtener_metadata_pdb(pdb_id):
    """Obtiene metadatos básicos de una entrada del PDB."""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    r = requests.get(url, timeout=15)
    if r.status_code == 200:
        return r.json()
    return {}

# Explorar la estructura 1IVO (EGFR con erlotinib co-cristalizado)
PDB_ID = '1IVO'
meta = obtener_metadata_pdb(PDB_ID)

if meta:
    struct = meta.get('struct', {})
    exptl = meta.get('exptl', [{}])[0]
    cell  = meta.get('cell', {})
    rcsb  = meta.get('rcsb_entry_info', {})

    print(f"ESTRUCTURA: {PDB_ID}")
    print("=" * 60)
    print(f"  Título:          {struct.get('title','N/A')[:65]}")
    print(f"  Método:          {exptl.get('method','N/A')}")
    print(f"  Resolución (Å):  {rcsb.get('resolution_combined', ['N/A'])[0] if rcsb.get('resolution_combined') else 'N/A'}")
    print(f"  Año deposición:  {meta.get('rcsb_accession_info',{}).get('initial_release_date','N/A')[:4]}")
    print(f"  Cadenas prot.:   {rcsb.get('polymer_entity_count_protein','N/A')}")
    print(f"  Ligandos:        {rcsb.get('nonpolymer_entity_count','N/A')}")
    print()
    print(f"  → Visualizar: https://www.rcsb.org/structure/{PDB_ID}")
    print(f"  → Descargar:  https://files.rcsb.org/download/{PDB_ID}.pdb")


In [ ]:
# ── Listar los ligandos co-cristalizados ────────────────────────────────────
def obtener_ligandos_pdb(pdb_id):
    """Obtiene los ligandos (moléculas pequeñas) en una estructura PDB."""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    r = requests.get(url, timeout=15)
    if r.status_code != 200:
        return []
    data = r.json()
    nonpoly = data.get('rcsb_entry_info', {}).get('nonpolymer_entity_count', 0)
    return nonpoly

# Obtener info de ligandos para varias estructuras de EGFR
print("ESTRUCTURAS DE EGFR CON LIGANDOS CO-CRISTALIZADOS")
print("=" * 60)

estructuras_interes = {
    '1IVO': 'EGFR + Erlotinib',
    '2ITY': 'EGFR + Gefitinib',
    '4LL0': 'EGFR + Osimertinib (3ra gen.)',
    '3W2S': 'EGFR mutante T790M',
    '1YY9': 'EGFR dominio extracelular',
}

for pdb_id, descripcion in estructuras_interes.items():
    meta = obtener_metadata_pdb(pdb_id)
    if meta:
        rcsb = meta.get('rcsb_entry_info', {})
        res = rcsb.get('resolution_combined', ['?'])[0] if rcsb.get('resolution_combined') else '?'
        metodo = (meta.get('exptl', [{}]) or [{}])[0].get('method', '?')[:10]
        n_lig = rcsb.get('nonpolymer_entity_count', '?')
        print(f"  {pdb_id}  {descripcion:<35} {metodo:<12} {res} Å  {n_lig} ligandos")


In [ ]:
# ── Descargar un archivo PDB directamente ───────────────────────────────────
def descargar_pdb(pdb_id, carpeta='.'):
    """Descarga el archivo PDB de la estructura indicada."""
    import os
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    r = requests.get(url, timeout=30)
    if r.status_code == 200:
        ruta = f"D:\\data_git\\curso_datascience\\files\\{pdb_id.upper()}.pdb"
        with open(ruta, 'w') as f:
            f.write(r.text)
        lineas = r.text.count('\n')
        atomos = r.text.count('\nATOM')
        hetams = r.text.count('\nHETATM')
        print(f"✅ {pdb_id.upper()}.pdb descargado")
        print(f"   Líneas totales:  {lineas}")
        print(f"   Registros ATOM:  {atomos}  (átomos de la proteína)")
        print(f"   Registros HETATM:{hetams}  (ligandos, agua, cofactores)")
        return ruta
    else:
        print(f"❌ Error al descargar {pdb_id}: HTTP {r.status_code}")
        return None

# Descargar EGFR + Erlotinib
ruta_pdb = descargar_pdb('1IVO')


In [ ]:
# ── Leer y parsear el PDB manualmente (sin librerías externas) ─────────────
# Esto ilustra cómo está estructurado un archivo PDB

if ruta_pdb:
    with open(ruta_pdb) as f:
        lineas = f.readlines()

    # Separar registros por tipo
    atomos    = [l for l in lineas if l.startswith('ATOM')]
    hetatm    = [l for l in lineas if l.startswith('HETATM')]
    header    = [l for l in lineas if l.startswith('HEADER')]
    compnd    = [l for l in lineas if l.startswith('COMPND')]
    remark    = [l for l in lineas if l.startswith('REMARK')]
    seqres    = [l for l in lineas if l.startswith('SEQRES')]

    print("ANÁLISIS DEL ARCHIVO PDB: 1IVO")
    print("=" * 50)
    print(f"  Registros HEADER:  {len(header)}")
    print(f"  Registros COMPND:  {len(compnd)}")
    print(f"  Registros REMARK:  {len(remark)}")
    print(f"  Registros SEQRES:  {len(seqres)}  (secuencia aminoacídica)")
    print(f"  Registros ATOM:    {len(atomos)}  (proteína)")
    print(f"  Registros HETATM:  {len(hetatm)} (ligandos + agua)")
    print()

    # Identificar los ligandos
    residuos_het = set()
    for linea in hetatm:
        resname = linea[17:20].strip()
        if resname not in ('HOH', 'WAT'):   # excluir agua
            residuos_het.add(resname)

    print(f"  Ligandos no-agua: {residuos_het}")
    print()
    print("Primeras 3 líneas ATOM (formato fijo del PDB):")
    for l in atomos[:3]:
        print(f"  {l.rstrip()}")
    print()
    print("Columnas del formato ATOM:")
    print("  1-6: tipo (ATOM/HETATM) | 7-11: nº átomo | 13-16: nombre átomo")
    print("  17-20: residuo | 22: cadena | 23-26: nº residuo | 31-54: coordenadas XYZ")


---
## 9. Cruce de bases de datos: ChEMBL + PubChem + PDB

En la práctica, nunca se usa una sola base de datos. El flujo típico de drug discovery  
conecta información de las tres fuentes.


In [ ]:
# ── Flujo completo para un target: EGFR ─────────────────────────────────────
print("FLUJO DE COLECCIÓN MULTI-BASE DE DATOS PARA EGFR")
print("=" * 60)
print()
print("PASO 1 — ChEMBL: identificar el target y sus inhibidores")
print("-" * 55)
print("  Target:      EGFR (CHEMBL203)")
print("  UniProt:     P00533")
print("  Inhibidores IC50 disponibles: >10.000 registros")
print()
print("PASO 2 — PubChem: enriquecer con propiedades y sinónimos")
print("-" * 55)
print("  Por cada compuesto de ChEMBL, obtener:")
print("  • CID de PubChem (identificador universal)")
print("  • Nombre IUPAC oficial")
print("  • Sinónimos y nombres comerciales")
print("  • Propiedades adicionales (TPSA, complejidad, carga)")
print()
print("PASO 3 — PDB: obtener estructuras para docking (Semana 6)")
print("-" * 55)
print("  Con UniProt P00533:")
print(f"  • {total} estructuras disponibles en el PDB")
print("  • Seleccionar la de mejor resolución con ligando co-cristalizado")
print("  • Descargar el PDB para docking en la semana 6")


In [ ]:
# ── Ejemplo: cruzar IDs de un compuesto entre bases de datos ───────────────
# Erlotinib tiene diferentes identificadores en cada base de datos:

print("IDENTIFICADORES CRUZADOS DE ERLOTINIB")
print("=" * 55)
print()
print("  Base de datos    ID                    URL")
print("  " + "-"*75)

ids_erlotinib = {
    'ChEMBL':   ('CHEMBL553',    'https://www.ebi.ac.uk/chembl/compound_report_card/CHEMBL553'),
    'PubChem':  ('CID 176870',   'https://pubchem.ncbi.nlm.nih.gov/compound/176870'),
    'DrugBank': ('DB00530',      'https://go.drugbank.com/drugs/DB00530'),
    'ChEBI':    ('CHEBI:114785', 'https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:114785'),
    'PDB lig.': ('FMM',          'https://www.rcsb.org/ligand/AQ4'),
    'InChIKey': ('AAKJLRGGTJKAMG-UHFFFAOYSA-N', '— identificador universal —'),
}

for db, (id_val, url) in ids_erlotinib.items():
    print(f"  {db:<12} {id_val:<25} {url[:55]}")

print()
print("💡 El InChIKey es el único identificador verdaderamente universal.")
print("   Permite cruzar compuestos entre CUALQUIER base de datos.")


In [ ]:
# ── Función utilitaria: colectar datos de un target nuevo ──────────────────
def colectar_target(nombre_target, organismo='Homo sapiens',
                    tipos_actividad=('IC50','Ki'), max_registros=500):
    """
    Función de alto nivel para colectar datos de un target desde ChEMBL.
    Devuelve un DataFrame con los datos de actividad.

    Parámetros:
        nombre_target    : nombre del target (ej. 'EGFR', 'CDK2', 'HIV protease')
        organismo        : organismo del target (default: humano)
        tipos_actividad  : tupla de tipos de actividad a descargar
        max_registros    : límite de registros por tipo de actividad
    """
    print(f"Colectando datos para: {nombre_target} ({organismo})")
    print("-" * 50)

    # 1. Buscar el target
    tc = new_client.target
    resultados = tc.filter(
        pref_name__icontains=nombre_target,
        organism=organismo,
        target_type='SINGLE PROTEIN'
    )
    df_t = pd.DataFrame.from_records(resultados)

    if len(df_t) == 0:
        print(f"  ❌ Target no encontrado: {nombre_target}")
        return None

    target_id = df_t.iloc[0]['target_chembl_id']
    target_nombre = df_t.iloc[0]['pref_name']
    print(f"  ✅ Target encontrado: {target_nombre} ({target_id})")

    # 2. Descargar actividades
    ac = new_client.activity
    dfs_actividad = []

    for tipo in tipos_actividad:
        datos = ac.filter(
            target_chembl_id=target_id,
            standard_type=tipo,
            assay_type='B'
        )
        df_tipo = pd.DataFrame.from_records(datos)
        if len(df_tipo) > 0:
            df_tipo = df_tipo.head(max_registros)
            dfs_actividad.append(df_tipo)
            print(f"  📥 {tipo}: {len(df_tipo)} registros")

    if not dfs_actividad:
        print("  ❌ Sin datos de actividad")
        return None

    df_final = pd.concat(dfs_actividad, ignore_index=True)

    # 3. Limpiar columnas esenciales
    columnas_esenciales = [
        'molecule_chembl_id', 'canonical_smiles',
        'standard_type', 'standard_value', 'standard_units',
        'pchembl_value', 'assay_chembl_id', 'confidence_score',
        'document_year', 'target_chembl_id'
    ]
    cols_disponibles = [c for c in columnas_esenciales if c in df_final.columns]
    df_final = df_final[cols_disponibles].copy()

    for col in ['standard_value', 'pchembl_value', 'confidence_score']:
        if col in df_final.columns:
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

    print(f"  📊 Total registros colectados: {len(df_final)}")
    print(f"  🧪 Compuestos únicos:          {df_final['molecule_chembl_id'].nunique()}")
    return df_final

# ── Probar la función con CDK2 ──────────────────────────────────────────────
df_cdk2 = colectar_target('Cyclin-dependent kinase 2', tipos_actividad=('IC50', 'Ki'))
if df_cdk2 is not None:
    print()
    print(df_cdk2.head(5)[['molecule_chembl_id','standard_type',
                             'standard_value','standard_units',
                             'pchembl_value']].to_string(index=False))


---
## 10. Ejercicio integrador: colecta datos para tu propio target

Este ejercicio conecta directamente con tu **proyecto final del curso**.  
Elige el target que identificaste en el NB-00 y colecta un dataset completo.


In [ ]:
# ══════════════════════════════════════════════════════
# 🎯 EJERCICIO INTEGRADOR — Colecta tu propio dataset
# ══════════════════════════════════════════════════════
#
# INSTRUCCIONES:
# 1. Cambia MI_TARGET por el nombre de tu target de interés
# 2. Ejecuta la celda y observa cuántos datos hay disponibles
# 3. Guarda el DataFrame — lo usarás en la Semana 3 (curación)
#
# Ejemplos de targets interesantes:
#   Oncología:         'CDK4', 'BRAF', 'KRAS', 'HER2', 'PARP1'
#   Enfermedades infec:'HIV protease', 'neuraminidase', 'Mpro'
#   Neurología:        'dopamine receptor D2', 'acetylcholinesterase'
#   Trop./Neglected:   'Plasmodium falciparum', 'Trypanosoma brucei'
#   Cardio:            'thrombin', 'factor Xa', 'renin'

MI_TARGET = 'Cyclin-dependent kinase 2'    # ← ¡Cambia esto por tu target!

# ── Paso 1: Colectar desde ChEMBL ──────────────────────────────────────────
df_mi_dataset = colectar_target(
    nombre_target=MI_TARGET,
    tipos_actividad=('IC50', 'Ki')
)

if df_mi_dataset is not None:
    print()
    print("Vista previa del dataset:")
    print(df_mi_dataset.head(5).to_string(index=False))


In [ ]:
# ── Paso 2: Estadísticas básicas del dataset colectado ─────────────────────
if df_mi_dataset is not None and len(df_mi_dataset) > 0:
    print(f"RESUMEN DEL DATASET: {MI_TARGET}")
    print("=" * 50)
    print(f"  Total registros:        {len(df_mi_dataset)}")
    print(f"  Compuestos únicos:      {df_mi_dataset['molecule_chembl_id'].nunique()}")

    if 'standard_type' in df_mi_dataset.columns:
        print(f"  Tipos de actividad:")
        for tipo, n in df_mi_dataset['standard_type'].value_counts().items():
            print(f"    {tipo}: {n}")

    if 'standard_units' in df_mi_dataset.columns:
        print(f"  Unidades encontradas:")
        for unidad, n in df_mi_dataset['standard_units'].value_counts().head(5).items():
            print(f"    {str(unidad)}: {n}")

    if 'pchembl_value' in df_mi_dataset.columns:
        validos = df_mi_dataset['pchembl_value'].dropna()
        if len(validos) > 0:
            print(f"  pActividad (n={len(validos)}):")
            print(f"    Mediana: {validos.median():.2f}")
            print(f"    Rango:   [{validos.min():.1f}, {validos.max():.1f}]")

    print()
    print("⚠️  ¿Hay múltiples unidades? → Semana 3: curación")
    print("⚠️  ¿Hay valores extremos?   → Semana 3: outliers")
    print("⚠️  ¿Hay duplicados?         → Semana 3: deduplicación")


In [ ]:
# ── Paso 3: Guardar el dataset ──────────────────────────────────────────────
if df_mi_dataset is not None and len(df_mi_dataset) > 0:
    nombre_archivo = f"dataset_{MI_TARGET.replace(' ','_').lower()}_raw.csv"
    df_mi_dataset.to_csv(nombre_archivo, index=False)
    print(f"✅ Dataset guardado: {nombre_archivo}")
    print()
    print("Este archivo lo usarás en la Semana 3 para curación de datos.")
    print()
    print("Columnas guardadas:")
    for col in df_mi_dataset.columns:
        no_nulos = df_mi_dataset[col].notna().sum()
        pct = no_nulos / len(df_mi_dataset) * 100
        print(f"  {col:<30} {no_nulos:>5}/{len(df_mi_dataset)} ({pct:.0f}% completo)")


---
## ✅ Resumen de lo aprendido

| Sección | Concepto clave |
|---------|---------------|
| **ChEMBL — targets** | Filtros por organismo, tipo, UniProt; confidence score de assays |
| **ChEMBL — actividades** | IC50 vs Ki vs EC50 vs Kd; binding vs functional; pActividad |
| **ChEMBL — compuestos** | Propiedades ADME, QED, fase clínica, mecanismo de acción, indicaciones |
| **ChEMBL — assays** | Confidence score, documentos fuente, tipos de ensayo |
| **ChEMBL — estructura** | Similitud Tanimoto, búsqueda por subestructura (scaffold) |
| **PubChem** | CID, IUPAC, sinónimos, API PUG-REST para batch |
| **PDB** | UniProt → PDB, metadatos de estructura, resolución, ligandos |
| **Cruce de fuentes** | InChIKey como puente universal, flujo ChEMBL→PubChem→PDB |
| **Dataset propio** | `colectar_target()` lista para usar en Semana 3 |

## 📅 Próximo notebook: NB-DATA-02 — Curación de datos moleculares

Con el dataset que acabas de colectar, en la **Semana 3** aprenderás a:
- Convertir y homogeneizar unidades (nM, µM, mg/mL...)
- Estandarizar SMILES con RDKit y ChEMBL Structure Pipeline
- Eliminar duplicados, sales y mezclas
- Clasificar compuestos como **activo / inactivo** con un umbral justificado

---
*NB-DATA-01 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*  
*Basado en: [intro_pharma_ai](https://github.com/FelPVic/intro_pharma_ai)*
